# MNIST From Scratch — full study on Colab

One-click reproduction of the entire NumPy-only MNIST pipeline.

Runtime expectation on a free CPU runtime:
- Main pack (Part A MLP 30 ep + Part B CNN 20 ep, full 50k): **~50 min**
- 5 ablation packs (optim/reg/aug + robust + error): **~80 min**
- Report rendering + PDF: **<1 min**

**Total: about 2-2.5 hours**, well within Colab's 12-hour session budget.

Outputs you get back:
- `MNIST_From_Scratch_Report_Leyan_Huang.pdf` (the final paper)
- `results_full.zip` (all JSON summaries, PNG figures, model checkpoints)

## 1. Clone the repository

In [ ]:
# Repository to clone (edit only if you fork it) ----------------------
REPO_URL = 'https://github.com/Lilllllllian/NNDL.git'
BRANCH   = 'main'
# ---------------------------------------------------------------------

import os, subprocess
if not os.path.isdir('/content/PJ1'):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, '/content/PJ1'])
%cd /content/PJ1
!git log -n 1 --pretty=format:'%h %s (%an, %ar)' && echo

## 2. Install dependencies (numpy + matplotlib are pre-installed on Colab)

In [ ]:
!pip install -q -r requirements.txt

## 3. Sanity check (gradient verification)
Confirms that the analytical and finite-difference gradients agree to ≲1e-8 for every layer.

In [ ]:
%cd /content/PJ1/codes
!python gradient_check.py

## 4. Part A + Part B — the headline MLP vs CNN comparison
Full 50k training set, 30 ep MLP, 20 ep CNN. ~50 min.

In [ ]:
!python -u run_full_study.py --pack main --epochs-mlp 30 --epochs-cnn 20

## 5. Direction 1 — Optimisation (5 settings, full 50k, 10 ep)

In [ ]:
!python -u run_full_study.py --pack optim --epochs-direction 10

## 6. Direction 2 — Regularisation (6 settings)

In [ ]:
!python -u run_full_study.py --pack reg --epochs-direction 10

## 7. Direction 3 — From-scratch geometric augmentation (8 settings)

In [ ]:
!python -u run_full_study.py --pack aug --epochs-direction 10

## 8. Direction 4 — Robustness (no training; reuses aug_clean and aug_RTS)

In [ ]:
!python -u run_full_study.py --pack robust

## 9. Direction 5 — Error and calibration analysis

In [ ]:
!python -u run_full_study.py --pack error

## 10. Render the final LaTeX report
`build_report.py` reads every `summary.json` and writes the .tex at the project root.

In [ ]:
!python build_report.py

## 11. Compile the PDF (Colab already ships with TeX Live via `apt`)

In [ ]:
# TeX Live is preinstalled in standard Colab; if not, uncomment the next line:
# !apt-get -qq install -y texlive-latex-recommended texlive-fonts-recommended texlive-latex-extra
%cd /content/PJ1
!pdflatex -interaction=nonstopmode -halt-on-error MNIST_From_Scratch_Report_Leyan_Huang.tex > /tmp/pdflatex1.log 2>&1 || tail -n 40 /tmp/pdflatex1.log
!pdflatex -interaction=nonstopmode -halt-on-error MNIST_From_Scratch_Report_Leyan_Huang.tex > /tmp/pdflatex2.log 2>&1 || tail -n 40 /tmp/pdflatex2.log
!ls -lh MNIST_From_Scratch_Report_Leyan_Huang.pdf

## 12. Download the deliverables

In [ ]:
import shutil
shutil.make_archive('/content/results_full', 'zip', '/content/PJ1/codes/results/full')
from google.colab import files
files.download('/content/PJ1/MNIST_From_Scratch_Report_Leyan_Huang.pdf')
files.download('/content/results_full.zip')

## 13. (Optional) Push results back to GitHub
Re-create a results branch with all generated assets so the repo carries a fully reproducible artefact set.

In [ ]:
# Configure your GitHub identity and a personal access token (PAT)
GIT_USER  = '<your-name>'
GIT_EMAIL = '<your-email>'
GIT_TOKEN = '<your-personal-access-token>'  # https://github.com/settings/tokens

%cd /content/PJ1
!git config user.name  "$GIT_USER"
!git config user.email "$GIT_EMAIL"
!git checkout -B colab-results
# Force-add the gitignored result folder so it lands in the branch:
!git add -f codes/results/full MNIST_From_Scratch_Report_Leyan_Huang.tex MNIST_From_Scratch_Report_Leyan_Huang.pdf
!git commit -m 'colab: full 50k training results + final PDF'
import re, subprocess
url = subprocess.check_output(['git','remote','get-url','origin']).decode().strip()
url_with_token = re.sub(r'https://', f'https://{GIT_TOKEN}@', url)
!git push --force {url_with_token} colab-results